# Experiment 04 — Tail leverage → Information gain

This notebook visualizes `result/table/04_tail_leverage_gaussian_vs_pareto.csv` produced by
`experiment/04_tail_leverage_gaussian_vs_pareto.py`.

The script estimates the upper-tail mean $m_G(lpha)$ (Prop. 12/13) and also exports an
*implied normalized information gain* using the paper's haystack rewrite (Eq. (8)):

$$\frac{\mathrm{Gain}}{I_{\mathrm{ver}}} \approx Bp + \sqrt{2\ln 2\,p(1-p)}\,m_G(\alpha)\,B\sqrt{J},\quad \alpha=B/K.$$


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
csv_path = ROOT / "result" / "table" / "04_tail_leverage_gaussian_vs_pareto.csv"
fig_path = ROOT / "result" / "figure" / "04_tail_leverage_gaussian_vs_pareto.pdf"

if not csv_path.exists():
    raise FileNotFoundError(f"Missing CSV: {csv_path}. Run the experiment script first.")

df = pd.read_csv(csv_path)

df_g = df[df["dist"] == "gaussian"].sort_values("alpha")
df_p = df[df["dist"].str.startswith("pareto")].sort_values("alpha")

# Parameters (exported by the .py script)
p = float(df["p"].iloc[0])
J = float(df["J"].iloc[0])
B = int(df["B"].iloc[0])
I_ver = float(df["I_ver"].iloc[0])
baseline = float(df["baseline_over_Iver"].iloc[0])  # = Bp
oracle = float(df["oracle_over_Iver"].iloc[0])      # = B

print(f"Using parameters: p={p}, J={J} bits, B={B}, I_ver={I_ver}")
print(f"Baseline Bp = {baseline:.4f}, Oracle = {oracle:.4f}")

plt.rcParams["pdf.fonttype"] = 42
plt.figure(figsize=(7, 4))

# x-axis: oversampling ratio 1/α = K/B
xg = df_g["oversampling_ratio"]
plt.plot(xg, df_g["gain_over_Iver_emp"], marker="o", label="Gaussian (empirical)")
plt.plot(xg, df_g["gain_over_Iver_exact"], linestyle="--", label="Gaussian (exact m_G)")
plt.plot(xg, df_g["gain_over_Iver_asympt"], linestyle=":", label="Gaussian asympt (Prop. 12)")

xp = df_p["oversampling_ratio"]
plt.plot(xp, df_p["gain_over_Iver_emp"], marker="o", label=f"{df_p['dist'].iloc[0]} (empirical)")
plt.plot(xp, df_p["gain_over_Iver_asympt"], linestyle=":", label="Pareto asympt (Prop. 13)")

# reference lines
plt.axhline(baseline, linestyle="--", linewidth=1, label="Random verification baseline Bp")
plt.axhline(oracle, linestyle="-", linewidth=1, label="Oracle upper bound B")

plt.xscale("log")
plt.xlabel("Oversampling ratio 1/α = K/B (log scale)")
plt.ylabel("Normalized gain Gain / I_ver")
plt.title("Tail leverage shown as information gain: Gaussian vs Pareto")
plt.legend()
plt.tight_layout()

fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, format="pdf")
plt.show()

print("Saved:", fig_path)
